## Basic Task：

#### Q-Learning For Maze Navigation Task

In [2]:
import numpy as np

#### Define a simple 6x6 Maze environment

- Deterministic transitions

- State space: 36 cells

- Actions: 4 directions (Left, Down, Right, Up)

- Invalid actions are not allowed (represented as NaN in R)

In [5]:
# 'S' = Start, 'G' = Goal, 'T' = Trap, 'X' = Obstacle, '.' = Free cell
maze = [
    ['S', '.', '.', '.', 'X', '.'],
    ['.', 'X', '.', '.', 'X', '.'],
    ['.', '.', '.', 'X', '.', '.'],
    ['.', 'X', '.', '.', '.', '.'],
    ['.', '.', 'X', '.', 'X', '.'],
    ['.', '.', '.', '.', 'T', 'G'],
]

# Maze size
H = 6
W = 6

# Action definitions
# 0 = Left, 1 = Down, 2 = Right, 3 = Up
A = [0, 1, 2, 3]
A_names = {0: 'Left', 1: 'Down', 2: 'Right', 3: 'Up'}

# Convert (row, col) to state id
def rc_to_s(r, c):
    return r * W + c

# Convert state id back to (row, col)
def s_to_rc(s):
    return divmod(s, W)

# Locate Special States
start_state = None
goal_state = None
trap_states = []
obstacle_states = []

for r in range(H):
    for c in range(W):
        s = rc_to_s(r, c)
        cell = maze[r][c]
        
        if cell == 'S':
            start_state = s
        elif cell == 'G':
            goal_state = s
        elif cell == 'T':
            trap_states.append(s)
        elif cell == 'X':
            obstacle_states.append(s)

print("Start state:", start_state)
print("Goal state:", goal_state)
print("Trap states:", trap_states)
print("Obstacle states:", obstacle_states)

Start state: 0
Goal state: 35
Trap states: [34]
Obstacle states: [4, 7, 10, 15, 19, 26, 28]


#### Transition function

In [6]:
# Move deltas for each action
move = {
    0: (0, -1),   # Left
    1: (1, 0),    # Down
    2: (0, 1),    # Right
    3: (-1, 0),   # Up
}

# Returns the next state if the action is valid.
# Returns None if the action is invalid (off-grid or into an obstacle).
def get_next_state(s, a):
    
    r, c = s_to_rc(s)
    dr, dc = move[a]
    r2, c2 = r + dr, c + dc

    # Off-grid -> invalid action
    if r2 < 0 or r2 >= H or c2 < 0 or c2 >= W:
        return None

    s2 = rc_to_s(r2, c2)

    # Into obstacle -> invalid action
    if s2 in obstacle_states:
        return None

    return s2

#### Reward function

- Goal: +10 and terminal
- Trap: -10 and terminal
- Normal move: -0.1 (encourage shorter paths)

In [9]:
# Reward is decided by the next state s2.
def get_reward_and_done(s2):
    
    if s2 == goal_state:
        return 10.0, True
    if s2 in trap_states:
        return -10.0, True
    return -0.1, False

#### Build the R table (NaN = invalid action)

In [11]:
num_states = H * W
num_actions = len(A)

# Initialize all as NaN (all actions invalid by default)
R = np.full((num_states, num_actions), np.nan)

for s in range(num_states):
    # Optional: no actions from terminal states
    if s == goal_state or s in trap_states:
        continue

    # Optional: obstacle cells are never entered, so no actions from them
    if s in obstacle_states:
        continue

    for a in A:
        s2 = get_next_state(s, a)
        if s2 is None:
            continue

        reward, done = get_reward_and_done(s2)
        R[s, a] = reward

print("R table shape:", R.shape)
print("R row for start_state:", R[start_state])
print("Available actions from start:", [A_names[a] for a in np.where(~np.isnan(R[start_state]))[0]])
print(R)

R table shape: (36, 4)
R row for start_state: [ nan -0.1 -0.1  nan]
Available actions from start: ['Down', 'Right']
[[  nan  -0.1  -0.1   nan]
 [ -0.1   nan  -0.1   nan]
 [ -0.1  -0.1  -0.1   nan]
 [ -0.1  -0.1   nan   nan]
 [  nan   nan   nan   nan]
 [  nan  -0.1   nan   nan]
 [  nan  -0.1   nan  -0.1]
 [  nan   nan   nan   nan]
 [  nan  -0.1  -0.1  -0.1]
 [ -0.1   nan   nan  -0.1]
 [  nan   nan   nan   nan]
 [  nan  -0.1   nan  -0.1]
 [  nan  -0.1  -0.1  -0.1]
 [ -0.1   nan  -0.1   nan]
 [ -0.1  -0.1   nan  -0.1]
 [  nan   nan   nan   nan]
 [  nan  -0.1  -0.1   nan]
 [ -0.1  -0.1   nan  -0.1]
 [  nan  -0.1   nan  -0.1]
 [  nan   nan   nan   nan]
 [  nan   nan  -0.1  -0.1]
 [ -0.1  -0.1  -0.1   nan]
 [ -0.1   nan  -0.1  -0.1]
 [ -0.1  -0.1   nan  -0.1]
 [  nan  -0.1  -0.1  -0.1]
 [ -0.1  -0.1   nan   nan]
 [  nan   nan   nan   nan]
 [  nan  -0.1   nan  -0.1]
 [  nan   nan   nan   nan]
 [  nan  10.    nan  -0.1]
 [  nan   nan  -0.1  -0.1]
 [ -0.1   nan  -0.1  -0.1]
 [ -0.1   nan  -0.1 